# Gold: patient_360

Builds `adb_amnuay.gold.patient_360` (one row per patient) from the silver layer per the STTM `sttm_silver_to_gold.xlsx`. Driving table `silver.patients`, LEFT JOINs to hospitals, insurance providers and lab results aggregated per patient. Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_amnuay.gold.patient_360
COMMENT 'Gold patient 360: one row per patient with hospital, insurance provider and lab test summary. Source: STTM silver to gold.'
AS
WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                   AS total_tests,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count,
    MAX(test_date)                             AS last_test_date
  FROM adb_amnuay.silver.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                   AS patient_id,
  CAST(p.first_name AS STRING)                                                AS first_name,
  CAST(p.last_name AS STRING)                                                 AS last_name,
  CAST(p.date_of_birth AS DATE)                                               AS date_of_birth,
  CAST(FLOOR(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  CAST(p.gender AS STRING)                                                    AS gender,
  CAST(p.city AS STRING)                                                      AS city,
  CAST(p.state AS STRING)                                                     AS state,
  CAST(p.registered_hospital_id AS INT)                                       AS hospital_id,
  CAST(h.hospital_name AS STRING)                                             AS hospital_name,
  CAST(p.insurance_provider_id AS INT)                                        AS insurance_provider_id,
  CAST(ip.provider_name AS STRING)                                            AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                     AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                             AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                             AS critical_test_count,
  CAST(l.last_test_date AS DATE)                                              AS last_test_date
FROM adb_amnuay.silver.patients p
LEFT JOIN adb_amnuay.silver.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_amnuay.silver.insurance_providers ip
  ON p.insurance_provider_id = ip.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id

In [0]:
# Return the number of records in the rebuilt gold table to the caller (e.g. a job or dbutils.notebook.run)
updated_df = spark.table("adb_amnuay.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))